In [1]:
import pandas as pd
import numpy as np
from unidecode import unidecode
import datetime as dt
import re as re

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('data_for_test_10000.csv')
df = df.drop(columns = 'Unnamed: 0')

In [3]:
df.head()

,ID,Fullname,Birthday,Sex,Hometown,Jobtype,Salary,Marital Status
0,123456789012,Nguyen Van A,12/09/1985,Male,Trà Vinh,Software Developer,15,Married
1,123456789012,Nguyen Van Anh,15/08/1990,Male,Quảng Nam,Software Engineer,500,Married
2,123456789012,Nguyen Van Anh,20/07/1985,Female,Hưng Yên,Architect,700,Married
3,123456789012,Trần Thị Minh Thu,12/05/1987,female,Thành phố Cần Thơ,Architect,200,married
4,123456789012,Nguyễn Văn A,20/03/1990,Male,Bà Rịa - Vũng Tàu,Architect,500,Single


In [4]:
df.isnull().sum()

ID                0
Fullname          0
Birthday          0
Sex               0
Hometown          0
Jobtype           0
Salary            0
Marital Status    0
dtype: int64

In [5]:
def find_number(text):
    num = re.findall(r'[0-9]+',text)
    return " ".join(num)

In [6]:
df['ID'] = df['ID'].apply(find_number)

index_dup = []
values_dup = []

for i, j in enumerate(df['ID'].duplicated()):
    if j == True:
        index_dup.append(i)
        values_dup.append(df['ID'].loc[i])

dup_val = []

for i in values_dup:
    if i not in dup_val:
        dup_val.append(i)

index_notdup = []
values_notdup = []

for i, j in enumerate(df['ID'].duplicated()):
    if j == False:
        index_notdup.append(i)
        values_notdup.append(df['ID'].loc[i])

def get_duplicate(column, duplicate_value):

    indexs = []
    values = []

    for i, j in enumerate(column.duplicated()):
        if j == True and column.loc[i] == str(duplicate_value):
            indexs.append(i)
            values.append(column.loc[i])

    return indexs, values

index_lst = []
value_lst = []
for i in dup_val:
    indexs, values = get_duplicate(df['ID'], i)
    index_lst.append(indexs)
    value_lst.append(values)

def escalate_value(list, list_index):

    new_values = []
    new_indexs = []
    for lst, index in zip(list, list_index):
        for i, j, k in zip(lst, range(len(lst)), index):
            new_indexs.append(k)
            new_values.append(int(i) + j + 1)

    return new_indexs, new_values

new_i, new_val = escalate_value(value_lst, index_lst)

new_es = pd.Series(data = new_val, index = new_i).to_frame()
not_dup_val = pd.Series(data = values_notdup, index = index_notdup).to_frame()
new_df = pd.concat([new_es, not_dup_val]).reset_index().sort_values(by = 'index', ascending = True).reset_index().drop(columns = ['index', 'level_0']).rename(columns = {0 : 'ID'})

df['ID'] = pd.to_numeric(new_df['ID'])

In [7]:
def remove_accents(input_str):
    return unidecode(input_str)

In [8]:
df['Fullname'] = df['Fullname'].apply(remove_accents)

In [9]:
df['Salary'] = pd.to_numeric(df['Salary'].apply(find_number))

In [10]:
df['Sex'] = df['Sex'].replace(['female', 'male'], ['Female', 'Male'])

In [11]:
df['Hometown'] = df['Hometown'].apply(remove_accents)

In [12]:
df['Marital Status'] = df['Marital Status'].replace(['married', 'Married}', 'Maried'], 'Married')
df['Marital Status'] = df['Marital Status'].replace(['single', ' Single'], 'Single')
df['Marital Status'] = df['Marital Status'].replace('divorced', 'Divorced')

In [13]:
df['Jobtype'] = df['Jobtype'].str.lower().str.strip()
jobtype_mapping = {'software developer': 'software engineer', 'architect': 'architect'}
df['Jobtype'] = df['Jobtype'].map(jobtype_mapping).fillna(df['Jobtype']).str.title()

In [14]:
for j, i in enumerate(df['Birthday']):
    if len(i) < 3:
        df = df.drop(index = j)
    elif len(i) > 2 and len(i) < 9:
        df = df.replace(i, i[:2] + '/' + i[2:4] + '/' + i[4:])
    elif len(i) < 10:
        split = i.split('/')
        split[1] = str(0) + split[1]
        new = split[0] + '/' + split[1] + '/' + split[2]
        df = df.replace(i, new)

for j, i in enumerate(df['Birthday']):
    if i.split('/')[0] == '31' and i.split('/')[1] == '02':
        df = df.drop(index = j).reset_index().drop(columns = 'index')

for j, i in enumerate(df['Birthday']):
    if len(i.split('/')[0]) == 4:
        split = i.split('/')
        split[0], split[2] = split[2], split[0]
        new = split[0] + '/' + split[1] + '/' + split[2]
        df = df.replace(i, new)

for j, i in enumerate(df['Birthday']):
    if len(i.split('-')[0]) in range(0, 10) and len(i.split('-')[1]) in range(0, 10) and len(i.split('-')[2]) in range(0, 10):
        split = i.split('-')
        new = split[0] + '/' + split[1] + '/' + split[2]
        df = df.replace(i, new)

In [15]:
df.to_csv('data_for_test_10000 NEW.csv', index = False)